# BioLBM in Agent Gym — Colab GPU checks

Select **Runtime → Change runtime type → GPU**. This notebook checks GPU functionality against a CPU Taylor–Green reference. The optional vessel run is an experimental viscous-fluid transient, not a water bioreactor validation.

Requires Python 3.12+ and CUDA-enabled PyTorch. Outputs are synthetic and local to this Colab session. No model API or GitHub token is required: Agent Gym is public. GPU/Colab execution is pending verification.


In [ ]:
import sys, subprocess, tempfile, os
from pathlib import Path
import torch
assert sys.version_info >= (3, 12), "Use a Python 3.12+ runtime for the pinned Lettuce version."
assert torch.cuda.is_available(), "Select a GPU runtime; refusing CPU fallback."
print("GPU:", torch.cuda.get_device_name(0), "Torch:", torch.__version__)
REPO = Path(tempfile.mkdtemp(prefix="agent-gym-", dir="/content"))
BRANCH = "feat/biolbm-module"  # Switch to main after merge, or pin a reviewed revision.
subprocess.check_call(["git", "clone", "--depth", "1", "--branch", BRANCH,
                       "https://github.com/jjbaloo2023/agent-gym.git", str(REPO)])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", f"{REPO}[biolbm]"])
sys.path.insert(0, str(REPO))
os.chdir(REPO)
RUNS = Path(tempfile.mkdtemp(prefix="biolbm-runs-", dir="/content"))
os.environ["AGENT_GYM_BIOLBM_RUNS"] = str(RUNS)
print("Revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 1. CPU/GPU agreement
Identical float64 2D cases; compare distributions and analytical velocity errors. This checks consistency, not grid convergence.


In [ ]:
import json, numpy as np
from agent_gym import biolbm_tools
config = dict(flow="taylor_green_2d", resolution=32, reynolds_number=20,
              mach_number=0.05, steps=100, report_interval=10, precision="float64")
print(json.dumps(biolbm_tools.validate_case("benchmark", {**config, "device":"cuda"}), indent=2))
cpu = biolbm_tools.run_case("benchmark", {**config, "device":"cpu"})
gpu = biolbm_tools.run_case("benchmark", {**config, "device":"cuda"})
for name, result in [("CPU", cpu), ("GPU", gpu)]:
    print(name, json.dumps(result, indent=2))
    assert result["status"]["state"] == "completed", result
    assert result["summary"]["benchmark_passed"], result
cpu_f = np.load(RUNS/cpu["run_id"]/"results"/"checkpoint.npz")["f"]
gpu_f = np.load(RUNS/gpu["run_id"]/"results"/"checkpoint.npz")["f"]
print("Maximum distribution difference:", np.max(np.abs(cpu_f-gpu_f)))
np.testing.assert_allclose(cpu_f, gpu_f, rtol=1e-7, atol=1e-10)


## 2. Larger GPU smoke test
D3Q19 periodic flow; 3D transient analytical validation is not claimed. Timing includes diagnostic/output overhead and is not a peak kernel benchmark.


In [ ]:
gpu3d = biolbm_tools.run_case("benchmark", dict(
    flow="taylor_green_3d", resolution=64, reynolds_number=40,
    mach_number=0.05, steps=200, report_interval=20,
    device="cuda", precision="float64"))
print(json.dumps(gpu3d, indent=2))
assert gpu3d["status"]["state"] == "completed", gpu3d


## 3. Optional synthetic 5 L vessel
3 L working liquid, 59.944 mm pitched paddle, 180 rpm CCW from above. Deliberately viscous fluid at 0.1 Pa·s; water at these coarse settings is rejected. CUDA is wired but unverified. Diffuse rotor forcing and short startup do not establish validated torque or mixing time.


In [ ]:
RUN_VESSEL = False  # Set True after the CPU/GPU benchmark checks pass.
if RUN_VESSEL:
    vessel = biolbm_tools.run_case("vessel", dict(
        viscosity_pa_s=0.1, spacing_m=0.003, lattice_tip_mach=0.1,
        rpm=180, steps=1200, report_interval=50, device="cuda", threads=2))
    print(json.dumps(vessel, indent=2))
    assert vessel["status"]["state"] == "completed", vessel
    from IPython.display import Image, display
    folder = RUNS/vessel["run_id"]/"results"
    display(Image(filename=str(folder/"geometry"/"geometry_and_lattice.png")))
    display(Image(filename=str(folder/"velocity_slices.png")))
    display(Image(filename=str(folder/"diagnostics.png")))


## Download results
Includes cases, status, logs, summaries, checkpoints, and VTK output for ParaView. Preserve the recorded repository revision and runtime versions with results.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/biolbm-results", "zip", str(RUNS))
files.download(archive)
